# MNESIS : boilerplate - loading all dependencies and variables



In [1]:
# %pip install -q -U -r requirements.txt

In [2]:
from pathlib import Path
data_cache = Path('../cached_data')
data_cache.mkdir(exist_ok=True)

figpath = None
figpath = Path('../figures')

In [3]:
RECOMPUTE = True
RECOMPUTE = False

DEBUG = 4 # to speed up simulations
DEBUG = 2 # to speed up simulations
DEBUG = 8 # to speed up simulations
DEBUG = 1 # production
if DEBUG > 1:
    print('running in debug mode with DEBUG =', DEBUG)


datetag = '2026-04-14' # AIROV
datetag = '2026-04-15' # CERCO
datetag = '2026-04-21' # WiP
datetag = '2026-05-18' # going stochastic
datetag = '2026-05-30' # going stochastic
datetag = '2026-06-10' # going Jean Zay
datetag = '2026-06-18' # getting ready for the camera ready
datetag = '2026-06-23' # getting ready for the camera ready
datetag = '2026-07-11' # new run with new parameters from the camera ready
print(f"datetag = '{datetag}'")

datetag = '2026-07-11'


In [4]:
# print('Files in that folder')
# %ls -ltra
# %ls -ltra {data_cache}

## loading libraries

In [5]:
import torch
import torch.nn as nn
from torch.optim.lr_scheduler import LambdaLR
from collections import OrderedDict
torch.set_float32_matmul_precision("medium")
torch.set_default_dtype(torch.float32)
torch.set_printoptions(precision=3, linewidth=140, sci_mode=False)
torch.autograd.set_detect_anomaly(True) # to DEBUG

if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"CUDA: {torch.cuda.is_available()}, Device: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device('cpu')
# device = torch.device('cpu') # uncomment to force CPU usage
print(f'Using device: {device}')

Using device: mps


In [6]:
import snntorch as snn
# from snntorch import surrogate
import snntorch.surrogate as surrogate

# from snntorch import functional as SF
from snntorch import utils as snn_utils
import snntorch.spikeplot as splt
print('SNNtorch version', snn.__version__)
i_pattern = 0

SNNtorch version 0.9.4


In [7]:
def pprint(str):
    print(len(str)*'=')
    print(str)
    print(len(str)*'=')

## handling figures

In [8]:
from tqdm import tqdm, trange
import numpy as np
phi = np.sqrt(5)/2 + 1/2
# import matplotlib
# matplotlib.use('Agg')      # head‑less, file‑only
import matplotlib.pyplot as plt
from matplotlib.figure import SubplotParams
subplotpars = SubplotParams(left=0.125, right=.95, bottom=0.25, top=.975, wspace=0.05, hspace=0.05,)

import seaborn as sns
import pandas as pd

def printfig(fig, name, fig_width, fig_height=None, exts=['pdf', 'png', 'svg'], figpath=figpath, dpi_exp=None, bbox='tight', verbose=True):
    if figpath is not None: 
        figpath.mkdir(exist_ok=True)
        if fig_height is None: fig_height = fig_width/phi
        cm = 1/2.54  # centimeters in inches
        fig.set_size_inches((fig_width*cm, fig_height*cm))  # Same as above
        for ext in exts:
            filename = figpath / f'{name}.{ext}'
            if verbose: print(f'Saving as {filename}')
            fig.savefig(filename, dpi=dpi_exp, bbox_inches=bbox, transparent=True)

## helper functions

In [9]:
def flip_bits(a, p_flip, seed=None, verbose=False):
    """
    Flip bits in a tensor with probability p_flip while preserving marginal frequency.
    
    Each bit is flipped independently with probability p_flip:
    - A zero has probability p_flip of being flipped to one
    - A one has probability p_flip of being flipped to zero
    
    The marginal frequency E[a] is exactly preserved: E[output] = E[a]
    while the temporal/spatial structure is stochastically modified.
    
    Args:
        a (torch.Tensor): Input tensor containing binary values (0s and 1s).
        p_flip (float): Probability of flipping each bit (between 0 and 1).
        
    Returns:
        torch.Tensor: New tensor with stochastically flipped bits.
    """
    generator = torch.Generator(device=a.device)
    if seed is None:
        # Use a non-deterministic seed when no seed is provided.
        seed = generator.seed()
    else:
        generator.manual_seed(seed)
    
    mask = torch.bernoulli(torch.ones_like(a) * p_flip, generator=generator)
    if verbose:
        print(f"Flipping {mask.sum().item()} bits out of {a.numel()} (p_flip={p_flip}, seed={seed}), {a.mean().item():.3e} -> {torch.where(mask == 1., 1 - a, a).mean().item():.3e}")
    flipped = torch.bernoulli(torch.ones_like(a) * a.mean(), generator=generator) # a stochastic bit flip
    return torch.where(mask == 1., flipped, a)

In [10]:
def stop(): assert False, "Temporary end of the road"

In [11]:
def approx_equals(series, value, rtol=1e-6, atol=1e-12):
    try:
        return np.isclose(series.astype(float), float(value), rtol=rtol, atol=atol)
    except (TypeError, ValueError):
        # Cas non numériques (bool, string, etc.)
        return series == value